# Datenanalyse mit SQL & Python - Tag 5: Abschlussprojekt & Storytelling mit Daten

**Freitag:** Vom Analyseergebnis zur überzeugenden Entscheidungsvorlage.  
**Storytelling-Daten:** Cleaned DS Jobs und Cleaned HR Data.  
**Abschlussprojekt:** Business Analytics Challenge - datenbasiertes Entscheiden in der Praxis.


## Lernziele

Nach Abschluss des Kurses können die Teilnehmenden:

- Daten aus verschiedenen Quellen abrufen, transformieren und kombinieren
- mit SQL Datenbanken abfragen und Analysen durchführen
- mit Python Daten bereinigen, analysieren und visualisieren
- statistische Grundlagen verstehen und anwenden
- datenbasiert argumentieren und Entscheidungsvorlagen erstellen
- Analyseergebnisse visuell und überzeugend präsentieren
- eine Datenstory als Vorlage für ein eigenes Abschlussprojekt strukturieren


## Themen des Tages

1. Vom Rohdaten zur Entscheidung: Analyseprozess Schritt für Schritt
1. Storytelling mit Daten: Zielgruppe, Frage, Kennzahl, Visualisierung, Empfehlung
1. Beispiel 1: Data-Science-Jobmarkt mit `Cleaned_DS_jobs2.csv`
1. Beispiel 2: HR-Analyse mit `cleaned_messy_HR_data.csv`
1. Nutzung von KI-gestützten Tools für Datenvisualisierung oder Präsentation
1. Abschlussprojekt: Business Analytics Challenge
1. Präsentation der Abschlussprojekte und Feedbackrunde


## Methoden & Tools

- SQL: SQLite für strukturierte Aggregationen
- Python: Pandas für Analyse, Matplotlib und Seaborn für Visualisierung
- Storytelling: Business-Frage, KPI, Erkenntnis, Empfehlung
- Visualisierung: klare Diagrammauswahl statt Diagramm-Sammlung
- Optional: KI-gestützte Unterstützung für Diagrammideen, Storyline und Präsentationsstruktur


## Ablauf

| Abschnitt | Fokus | Ergebnis |
|---|---|---|
| Einstieg | Von Analyse zu Entscheidung | Storytelling-Framework |
| Beispiel 1 | Cleaned DS Jobs | Jobmarkt-Storyline |
| Beispiel 2 | Cleaned HR Data | HR-Storyline |
| Storytelling | Erkenntnisse strukturieren | Präsentationsbausteine |
| KI-Tools | Unterstützung reflektiert nutzen | bessere Story-Entwürfe |
| Abschlussprojekt | Business Analytics Challenge | Analyse + Empfehlung |
| Präsentation | Ergebnisse vorstellen | Feedback und Kursabschluss |


## Einrichtung & Importe

Wir nutzen zwei bereits bereinigte Datensätze aus dem Repository `eyowhite/Messy-dataset`:

- `Cleaned_DS_jobs2.csv`
- `cleaned_messy_HR_data.csv`

Die Datensätze sind bewusst **cleaned**, damit der Vormittag auf Storytelling fokussiert. Am Nachmittag wenden die Gruppen denselben Ablauf auf messy data an.


In [ ]:
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', 100)

DS_JOBS_URL = 'https://raw.githubusercontent.com/eyowhite/Messy-dataset/main/Cleaned_DS_jobs2.csv'
HR_CLEAN_URL = 'https://raw.githubusercontent.com/eyowhite/Messy-dataset/main/cleaned_messy_HR_data.csv'

ds_jobs = pd.read_csv(DS_JOBS_URL, engine='python', on_bad_lines='warn')
hr_clean = pd.read_csv(HR_CLEAN_URL, engine='python', on_bad_lines='warn')

print('ds_jobs:', ds_jobs.shape)
print('hr_clean:', hr_clean.shape)


## 1. Storytelling-Framework

Eine gute Datenstory ist keine Liste von Diagrammen. Sie folgt einer Entscheidung:

1. **Zielgruppe:** Wer entscheidet?
1. **Business-Frage:** Welche Entscheidung soll unterstützt werden?
1. **KPI:** Welche Kennzahl ist zentral?
1. **Analyse:** Welche Daten zeigen das?
1. **Visualisierung:** Welches Diagramm macht die Aussage klar?
1. **Empfehlung:** Was sollte getan werden?
1. **Grenzen:** Was wissen wir noch nicht?

Dieses Framework ist die Vorlage für das Nachmittagsprojekt.


## 2. Beispiel 1: Data-Science-Jobmarkt

**Business-Frage:**

> Welche Faktoren hängen mit höheren Data-Science-Gehältern zusammen, und worauf sollte ein Jobseeker oder Recruiting-Team achten?

Mögliche KPIs:

- durchschnittliches Gehalt
- Anzahl der Job Postings
- Gehalt nach Standort
- Gehalt nach Sektor oder Branche
- Gehalt nach Rating oder Firmenalter


In [ ]:
ds_jobs.head()


In [ ]:
print(ds_jobs.columns.tolist())
ds_jobs.describe(include='all').T.head(20)


### 2.1 SQL: Gehalt nach Standort

Wir nutzen SQL, um eine klare Reporting-Tabelle zu erstellen.


In [ ]:
conn = sqlite3.connect(':memory:')
ds_jobs.to_sql('ds_jobs', conn, index=False, if_exists='replace')

# Spaltennamen unterscheiden sich je nach cleaned dataset leicht.
# Diese Hilfsfunktion sucht passende Spalten.
def find_col(df, candidates):
    normalized = {c.lower().replace(' ', '_'): c for c in df.columns}
    for candidate in candidates:
        key = candidate.lower().replace(' ', '_')
        if key in normalized:
            return normalized[key]
    return None

salary_col = find_col(ds_jobs, ['avg_salary', 'avg_salary_k', 'average_salary', 'salary'])
location_col = find_col(ds_jobs, ['location', 'job_location', 'state'])
sector_col = find_col(ds_jobs, ['sector', 'industry'])
rating_col = find_col(ds_jobs, ['rating'])

print('salary_col:', salary_col)
print('location_col:', location_col)
print('sector_col:', sector_col)
print('rating_col:', rating_col)


In [ ]:
if salary_col and location_col:
    query = f'''
    SELECT
        "{location_col}" AS location,
        COUNT(*) AS postings,
        AVG("{salary_col}") AS avg_salary
    FROM ds_jobs
    WHERE "{salary_col}" IS NOT NULL
      AND "{location_col}" IS NOT NULL
    GROUP BY "{location_col}"
    HAVING COUNT(*) >= 3
    ORDER BY avg_salary DESC;
    '''
    salary_by_location = pd.read_sql_query(query, conn)
else:
    salary_by_location = pd.DataFrame()

salary_by_location.head(10)


### 2.2 Python EDA: Gehaltsverteilung und Sektorvergleich

Python hilft, die SQL-Ergebnisse zu visualisieren und weitere Fragen zu prüfen.


In [ ]:
if salary_col:
    plt.hist(ds_jobs[salary_col].dropna(), bins=25)
    plt.title('Verteilung der Gehälter im DS-Jobmarkt')
    plt.xlabel('Gehalt')
    plt.ylabel('Anzahl Job Postings')
    plt.show()
else:
    print('Keine Gehaltsspalte gefunden. Bitte Spaltennamen prüfen.')


In [ ]:
if salary_col and sector_col:
    sector_report = (
        ds_jobs
        .dropna(subset=[salary_col, sector_col])
        .groupby(sector_col)[salary_col]
        .agg(['count', 'mean', 'median'])
        .query('count >= 3')
        .sort_values('mean', ascending=False)
    )
    display(sector_report.head(10))

    sns.barplot(data=sector_report.head(10).reset_index(), x='mean', y=sector_col)
    plt.title('Durchschnittsgehalt nach Sektor')
    plt.xlabel('Durchschnittsgehalt')
    plt.ylabel('Sektor')
    plt.show()
else:
    print('Für den Sektorvergleich fehlen passende Spalten.')


### 2.3 Datenstory: Jobmarkt

Beispiel-Storyline:

- **Zielgruppe:** Jobseeker oder Recruiting-Team
- **Business-Frage:** Welche Faktoren hängen mit höheren Gehältern zusammen?
- **KPI:** Durchschnittliches Gehalt
- **Finding:** Gehälter unterscheiden sich nach Standort und Sektor
- **Empfehlung:** Standort und Sektor sollten bei Gehaltsbenchmarking gemeinsam betrachtet werden
- **Grenze:** Jobtitel, Seniorität und Unternehmensgröße können die Aussage verzerren


## 3. Beispiel 2: HR-Analyse

**Business-Frage:**

> Welche HR-Gruppen zeigen auffällige Muster bei Gehalt, Zufriedenheit, Performance oder möglichem Fluktuationsrisiko?

Mögliche KPIs:

- durchschnittliches Gehalt
- Zufriedenheit
- Performance Score
- Beschäftigungsdauer
- Abteilung oder Rolle
- Fluktuationsindikator, falls vorhanden


In [ ]:
hr_clean.head()


In [ ]:
print(hr_clean.columns.tolist())
hr_clean.describe(include='all').T.head(20)


### 3.1 HR-Spalten für Analyse auswählen

Da HR-Datensätze unterschiedliche Spaltennamen haben können, suchen wir passende Spalten dynamisch.


In [ ]:
salary_hr_col = find_col(hr_clean, ['salary', 'monthly_income', 'income', 'gehalt'])
department_col = find_col(hr_clean, ['department', 'dept', 'abteilung'])
satisfaction_col = find_col(hr_clean, ['satisfaction', 'job_satisfaction', 'employee_satisfaction'])
performance_col = find_col(hr_clean, ['performance', 'performance_score', 'rating'])
attrition_col = find_col(hr_clean, ['attrition', 'left', 'turnover', 'fluktuation'])

print('salary_hr_col:', salary_hr_col)
print('department_col:', department_col)
print('satisfaction_col:', satisfaction_col)
print('performance_col:', performance_col)
print('attrition_col:', attrition_col)


### 3.2 HR-Analyse mit Pandas

Wir erstellen eine Gruppentabelle, die als Grundlage für eine Story dienen kann.


In [ ]:
numeric_cols = hr_clean.select_dtypes(include='number').columns.tolist()
category_cols = hr_clean.select_dtypes(include=['object', 'string']).columns.tolist()

print('Numerische Spalten:', numeric_cols[:10])
print('Kategoriale Spalten:', category_cols[:10])


In [ ]:
# Fallback, falls die erwarteten Spaltennamen anders heißen.
value_col = salary_hr_col if salary_hr_col else (numeric_cols[0] if numeric_cols else None)
group_col = department_col if department_col else (category_cols[0] if category_cols else None)

if value_col and group_col:
    hr_group_report = (
        hr_clean
        .groupby(group_col)[value_col]
        .agg(['count', 'mean', 'median'])
        .sort_values('mean', ascending=False)
    )
    display(hr_group_report.head(10))
else:
    hr_group_report = pd.DataFrame()
    print('Keine passenden Spalten für Gruppenanalyse gefunden.')


In [ ]:
if not hr_group_report.empty:
    sns.barplot(data=hr_group_report.head(10).reset_index(), x='mean', y=group_col)
    plt.title(f'Durchschnitt von {value_col} nach {group_col}')
    plt.xlabel(value_col)
    plt.ylabel(group_col)
    plt.show()


### 3.3 Datenstory: HR

Beispiel-Storyline:

- **Zielgruppe:** HR-Leitung oder People Operations
- **Business-Frage:** Welche Gruppen brauchen Aufmerksamkeit?
- **KPI:** abhängig vom Datensatz, z. B. Gehalt, Zufriedenheit, Performance oder Fluktuation
- **Finding:** Bestimmte Gruppen unterscheiden sich sichtbar in der gewählten Kennzahl
- **Empfehlung:** HR sollte diese Gruppe genauer prüfen und gezielte Maßnahmen testen
- **Grenze:** Ohne Kontext zu Rolle, Seniorität und Arbeitszeit ist die Interpretation vorsichtig zu formulieren


## 4. Von Beispiel zu Projekt: Vorlage zum Nachmachen

Die Nachmittagsgruppen können diese Struktur übernehmen:

1. **Business-Rolle:** formuliert Ziel, Entscheidung, KPIs
1. **Data-Analyst-Rolle:** prüft Daten, reinigt, analysiert, visualisiert
1. **SQL:** erstellt 1-2 aggregierte Reporting-Tabellen
1. **Python EDA:** erstellt 2-3 Diagramme
1. **Story:** Findings, Empfehlung, Grenzen, nächster Schritt

Die Beispiele mit DS Jobs und HR Data zeigen bewusst: Erst Frage und Zielgruppe, dann Code.


## 5. KI-gestützte Tools reflektiert nutzen

KI-Tools können unterstützen bei:

- Formulierung einer klaren Analysefrage
- Auswahl passender Diagrammtypen
- Struktur einer Präsentation
- sprachlicher Verdichtung von Erkenntnissen

Wichtig: KI ersetzt nicht die fachliche Prüfung. Zahlen, Annahmen und Interpretationen müssen immer von dir kontrolliert werden.


## Abschlussprojekt: Business Analytics Challenge

**Aufgabe für den Nachmittag:** Entwickle eine datenbasierte Entscheidungsvorlage mit einem messy Dataset.

Deine Präsentation sollte enthalten:

1. Fragestellung und Zielgruppe
1. kurze Datenbeschreibung und Datenqualität
1. 2-3 zentrale Analysen
1. 2-3 passende Visualisierungen
1. klare Erkenntnisse
1. konkrete Handlungsempfehlung
1. Grenzen der Analyse und nächster Schritt


## Präsentationsstruktur

Eine einfache Storyline:

1. **Ausgangslage:** Was ist die Business-Frage?
1. **Datenbasis:** Welche Daten wurden genutzt?
1. **Analyse:** Was wurde untersucht?
1. **Erkenntnisse:** Was fällt auf?
1. **Empfehlung:** Was sollte entschieden oder ausprobiert werden?
1. **Nächster Schritt:** Welche Daten oder Tests wären als Nächstes sinnvoll?


## Zusammenfassung

Tag 5 verbindet alle Kursteile:

- SQL für strukturierte Abfragen
- Python/Pandas für Analyse und Visualisierung
- Data Cleaning als Voraussetzung für belastbare Ergebnisse
- Storytelling für Entscheidungen
- Präsentation für Wirkung

Die Beispiele `Cleaned_DS_jobs2.csv` und `cleaned_messy_HR_data.csv` dienen als Schablone für das eigene Gruppenprojekt.
